In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Medications dispensed from the Pyxis automated cabinet for ED-only patients
# (disposition = 'HOME', hadm_id IS NULL — not admitted to the hospital).
# Pyxis dispenses map to the administer_medication action — the only medication
# action available during an ED stay (no drips, rate changes, or stops in ED).
# Note: Dispensing from Pyxis does not guarantee administration;
# nurses may return medications. This is the best available signal for ED-only patients.

query_ed_only_meds = """
SELECT
  e.subject_id,
  e.stay_id        AS ed_stay_id,
  e.hadm_id,
  e.disposition,
  p.charttime,
  p.med_rn,
  p.name           AS medication,
  TRUE             AS in_er
FROM `physionet-data.mimiciv_ed.edstays` e
JOIN `physionet-data.mimiciv_ed.pyxis` p
  ON e.stay_id = p.stay_id
WHERE e.disposition = 'HOME'
  AND e.hadm_id IS NULL
  AND p.charttime BETWEEN e.intime AND e.outtime
ORDER BY e.subject_id, e.stay_id, p.charttime
"""

print("Running ED-only meds query...")
df_ed_only_meds = client.query(query_ed_only_meds).to_dataframe()
print(f"Shape: {df_ed_only_meds.shape}")
print(f"Unique patients: {df_ed_only_meds['subject_id'].nunique():,}")
print(f"Unique ED stays: {df_ed_only_meds['ed_stay_id'].nunique():,}")
print(f"\nTop medications:\n{df_ed_only_meds['medication'].value_counts().head(10)}")
df_ed_only_meds.head()

In [ ]:
DESCRIPTION = (
    "Medications administered during the ED stay for ED-only patients (not admitted to hospital). "
    "Derived from ed.pyxis (Pyxis dispense records). "
    "Mapped to administer_medication action — the only medication action available during an ED stay."
)

ds = Dataset.from_pandas(df_ed_only_meds, split='ed_only')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="ed_only_meds", data_dir="stay_medications")
push_dataset_card("ADS599-Capstone/raw_data", config_name="ed_only_meds", split="ed_only", data_dir="stay_medications", description=DESCRIPTION)
print("Pushed ed_only_meds to HuggingFace Hub.")